[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/filippolonghi/medleydb-mert-instrument-classification/blob/main/notebooks/02_mert_representation_ablation.ipynb)


# 02 - MERT Representation, Pooling, and Model Size Ablation

## Research question

Which MERT layer or layer aggregation gives the most useful isolated-stem instrument representation, and is the larger MERT variant worth its cost?

## Approach

The classifier head is fixed to an MLP with hidden_dim=256 and dropout=0.2. Each layer/pooling/model-size setting has a separate cache directory, and the extractor validates cache metadata before reuse.

## What is fixed

Unless a selected config says otherwise: MedleyDB instrument labels, the `largest_balanced` split, MERT-v1-95M, 5 s segments, validation-based model selection, and test metrics saved only after training.

## What is varied

Layer 6, layer 9, last layer, last3avg, last6avg, all-layer learned softmax weighting, max pooling, meanmax pooling, MERT-v1-95M versus MERT-v1-330M.

## Expected interpretation

Representation changes should be compared by validation macro-F1 first, then test macro-F1 and per-class behavior for the selected candidate.


## What you can change

- `PROJECT_ROOT`, `RUN_ROOT`, and `MEDLEYDB_ROOT` for local vs Colab/Drive paths.
- `EXPERIMENT_GROUPS`, `DATASET_GROUPS`, `MIXTURE_GROUPS`, and `SELECTED_EXPERIMENTS` to run one config at a time.
- `REPLACE_EXISTING = True` only when intentionally overwriting a completed run.


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys
import yaml
import pandas as pd

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/content/medleydb-mert-instrument-classification"))
RUN_ROOT = Path(os.environ.get("RUN_ROOT", "/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1"))
MEDLEYDB_ROOT = Path(os.environ.get("MEDLEYDB_ROOT", "/content/drive/MyDrive/medleydb_mert_project/MedleyDB"))

os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["RUN_ROOT"] = str(RUN_ROOT)
os.environ["MEDLEYDB_ROOT"] = str(MEDLEYDB_ROOT)

SUBSET_PROFILE = "largest_balanced"
LABEL_GRANULARITY = "medleydb_instrument"
RESULTS_DIR = RUN_ROOT / "results"
CHECKPOINTS_DIR = RUN_ROOT / "checkpoints"
METADATA_DIR = RUN_ROOT / "data" / "metadata"
CACHE_ROOT = RUN_ROOT / "data" / "cache"
REPORTS_DIR = RUN_ROOT / "data" / "reports"
for path in [RESULTS_DIR, CHECKPOINTS_DIR, METADATA_DIR, CACHE_ROOT, REPORTS_DIR, RUN_ROOT / "configs"]:
    path.mkdir(parents=True, exist_ok=True)

if PROJECT_ROOT.exists():
    os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Run root:", RUN_ROOT)
print("MedleyDB root:", MEDLEYDB_ROOT)
print("Subset profile:", SUBSET_PROFILE)
print("Label granularity:", LABEL_GRANULARITY)


In [ ]:
def q(path):
    return shlex.quote(str(path))


def run(cmd, check=True):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, check=check)


def load_config(config_path):
    return yaml.safe_load(Path(config_path).read_text(encoding="utf-8"))


def run_experiment_config(config_path, *, extract_embeddings=True, replace_existing=False):
    config_path = Path(config_path)
    cfg = load_config(config_path)
    approach = cfg.get("approach", "frozen_embeddings")
    if approach == "frozen_embeddings" and extract_embeddings:
        run(f"python -m src.features.extract_mert_embeddings --experiment-config {q(config_path)} --batch-size 1 --device auto")
    elif approach == "polyphonic_multilabel" and extract_embeddings:
        run(f"python -m src.features.extract_mert_mixture_embeddings --experiment-config {q(config_path)} --batch-size 1 --device auto")
    args = f"--config {q(config_path)}"
    if replace_existing:
        args += " --replace-existing"
    return run(f"python -m src.experiments.run_experiment {args}")


def run_selected(configs, *, extract_embeddings=True, replace_existing=False):
    for config in configs:
        run_experiment_config(config, extract_embeddings=extract_embeddings, replace_existing=replace_existing)


## Experiment Groups

Run one representation config at a time unless the needed caches already exist. Different layer, pooling, segment duration, or MERT model size means a different cache.

In [ ]:
EXPERIMENT_GROUPS = {
    "layer_ablation": [
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_layer6_mean_mlp_h256_d02.yaml",
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_layer9_mean_mlp_h256_d02.yaml",
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02.yaml",
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last3avg_mean_mlp_h256_d02.yaml",
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last6avg_mean_mlp_h256_d02.yaml",
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_all_weighted_mean_mlp_h256_d02.yaml",
    ],
    "pooling_ablation": [
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_max_mlp_h256_d02.yaml",
        "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_meanmax_mlp_h256_d02.yaml",
    ],
    "model_size_ablation": [
        "configs/experiments/isolated_largest_balanced_medleydb_mert330_last_mean_mlp_h256_d02.yaml",
        "configs/experiments/isolated_largest_balanced_medleydb_mert330_all_weighted_mean_mlp_h256_d02.yaml",
    ],
}
SELECTED_EXPERIMENTS = [
    "configs/experiments/isolated_largest_balanced_medleydb_mert95_layer6_mean_mlp_h256_d02.yaml",
]
REPLACE_EXISTING = False

run_selected(SELECTED_EXPERIMENTS, extract_embeddings=True, replace_existing=REPLACE_EXISTING)


In [ ]:
registry_path = RESULTS_DIR / "experiment_registry.csv"
if registry_path.exists():
    wanted = [Path(p).stem for p in sum(EXPERIMENT_GROUPS.values(), [])]
    display(pd.read_csv(registry_path).query("experiment_id in @wanted"))
else:
    print("No registry yet:", registry_path)
